# 06 — LV-Eval FactRecall Reproduction
Reproduce the AHN-GDN FactRecall baseline exactly before debugging.

In [1]:
# Environment
import sys, torch, transformers, datasets, flash_attn, fla

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("FlashAttention:", flash_attn.__version__)
print("FLA: OK")

Python: 3.12.14
Torch: 2.11.0+cu128
CUDA: 12.8
GPU: NVIDIA A100-SXM4-40GB
Transformers: 4.51.0
Datasets: 3.6.0
FlashAttention: 2.8.3
FLA: OK


In [2]:
# Baseline configuration
MODEL_PATH = "../merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN"
MODEL_NAME = "Qwen-2.5-Instruct-3B-AHN-GDN"
DATASET = "factrecall_en_128k"

METHOD = "ahn"
START_SIZE = 128
RECENT_SIZE = 8064
MAX_LENGTH = 15500

print({
    "model": MODEL_PATH,
    "dataset": DATASET,
    "method": METHOD,
    "start_size": START_SIZE,
    "recent_size": RECENT_SIZE,
    "max_length": MAX_LENGTH,
})

{'model': '../merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN', 'dataset': 'factrecall_en_128k', 'method': 'ahn', 'start_size': 128, 'recent_size': 8064, 'max_length': 15500}


In [7]:
# Baseline reproduction — original failing condition
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "eval/lveval"))

from utils import (
    load_model_and_tokenizer_once,
    load_LVEval_dataset,
    truncate_prompt,
    build_chat,
    model_generate,
    post_process,
)
from config import DATASET_PROMPT, DATASET_MAXGEN, DATASET_METRIC

TASK = "factrecall_en"
DATASET = "factrecall_en_128k"
MODEL_PATH = str(ROOT / "merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN")

# Original failing configuration
BASELINE_N = 30
BASELINE_MAX_LENGTH = 15500
BASELINE_RECENT_SIZE = 8064
START_SIZE = 128
NAME = "qwen2"

MAX_GEN = DATASET_MAXGEN[TASK]
score_fn = DATASET_METRIC[TASK]

baseline_model, baseline_tok = load_model_and_tokenizer_once(
    0,
    MODEL_PATH,
    method="ahn",
    start_size=START_SIZE,
    recent_size=BASELINE_RECENT_SIZE,
)

data = load_LVEval_dataset(DATASET, None)
baseline_results = []

for i, ex in enumerate(data[:BASELINE_N]):
    prompt = DATASET_PROMPT[TASK].format(**ex)
    prompt = truncate_prompt(baseline_tok, prompt, BASELINE_MAX_LENGTH)
    prompt = build_chat(baseline_tok, prompt, NAME)

    pred = model_generate(
        baseline_tok, prompt, MAX_GEN, baseline_model
    )
    pred = post_process(pred, NAME)

    f1 = max(score_fn(pred, a) for a in ex["answers"])

    baseline_results.append(f1)
    print(f"{i:02d} | F1={f1:.4f} | {pred!r}")

mean_f1 = sum(baseline_results) / len(baseline_results)

print("\n=== BASELINE SUMMARY ===")
print("Examples:", BASELINE_N)
print("Recent size:", BASELINE_RECENT_SIZE)
print("Max length:", BASELINE_MAX_LENGTH)
print("Mean F1:", mean_f1)
print("F1 x100:", mean_f1 * 100)

using device cuda:0
cuda:0


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loading dataset >>>>>>>>> factrecall_en_128k
00 | F1=0.0000 | 'The name of the scientist widely acclaimed as the foundational figure of modern physics, according'
01 | F1=0.0000 | 'Albert Einstein is widely acclaimed as the foundational figure of modern physics.'
02 | F1=0.0000 | 'Albert Einstein is widely acclaimed as the foundational figure of modern physics.'
03 | F1=0.0000 | 'The article does not contain information about a scientist widely acclaimed as the foundational figure of'
04 | F1=0.0000 | 'Albert Einstein is widely acclaimed as the foundational figure of modern physics.'
05 | F1=0.0000 | 'Albert Einstein is widely acclaimed as the foundational figure of modern physics.'
06 | F1=0.0000 | 'The article does not provide information about a scientist widely acclaimed as the foundational figure of'
07 | F1=0.0000 | 'The name of the scientist widely acclaimed as the foundational figure of modern physics is Albert'
08 | F1=0.0000 | 'Albert Einstein is widely acclaimed as the found

## Result

Baseline successfully reproduced on a fresh A100 instance.

- Dataset: `factrecall_en_128k`
- Examples: 30
- Method: AHN
- `start_size=128`
- `recent_size=8064`
- Prompt max length: 15,500 tokens
- Official generation cap: 16 tokens
- Official scorer: `qa_f1_score`
- Mean F1: **0.0**
- Reported score: **0.0**

This notebook records the baseline only. No debugging interventions or altered retention-window experiments are included.


### 50-example FactRecall validation

Evaluation uses the corrected LV-Eval configuration:

- AHN-GDN checkpoint
- `start_size=128`
- `recent_size=32640`
- `max_length=256000`
- `NAME="qwen2"` for the upstream Qwen chat template
- official `max_new_tokens=16`
- official FactRecall F1 scorer

Fifty examples are sampled evenly across the 200-example dataset to avoid the position bias of `data[:N]`.

In [8]:
# 50-example validation
TASK = "factrecall_en"
NAME = "qwen2"
MODEL = "../merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN"

data = load_LVEval_dataset("factrecall_en_128k", None)
indices = [round(i * (len(data) - 1) / 49) for i in range(50)]

model, tok = load_model_and_tokenizer_once(
    0, MODEL, method="ahn", start_size=128, recent_size=32640
)

scores = []

for i in indices:
    ex = data[i]
    prompt = DATASET_PROMPT[TASK].format(**ex)
    prompt = truncate_prompt(tok, prompt, 256000)
    prompt = build_chat(tok, prompt, NAME)

    x = tok(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        y = model.generate(
            **x,
            max_new_tokens=DATASET_MAXGEN[TASK],
            do_sample=False,
            num_logits_to_keep=1,
        )[0]

    pred = tok.decode(
        y[x.input_ids.shape[-1]:],
        skip_special_tokens=True,
    )
    pred = post_process(pred, NAME)

    score = max(
        DATASET_METRIC[TASK](pred, gold)
        for gold in ex["answers"]
    )
    scores.append(score)

    print(f"{i:3d} | F1={score:.4f} | {pred}")

print("\nNonzero:", sum(s > 0 for s in scores), "/ 50")
print("Mean F1:", sum(scores) / len(scores))
print("F1 x100:", 100 * sum(scores) / len(scores))

loading dataset >>>>>>>>> factrecall_en_128k
using device cuda:0
cuda:0


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (32768). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


  0 | F1=0.0000 | Albert Einstein


/venv/main/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/venv/main/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/venv/main/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  4 | F1=0.0000 | The scientist widely acclaimed as the foundational figure of modern physics is Newton, but the
  8 | F1=0.0000 | David Beckham
 12 | F1=0.0000 | David Beckham
 16 | F1=0.0000 | David Beckham
 20 | F1=0.0000 | The article mentions a scientist widely acclaimed as the foundational figure of modern physics, but
 24 | F1=0.0000 | John Beverley
 28 | F1=0.0000 | Albert Einstein
 32 | F1=0.0000 | The name of the scientist widely acclaimed as the foundational figure of modern physics is Albert
 37 | F1=0.0000 | John Beverley
 41 | F1=0.0000 | Albert Einstein is widely acclaimed as the foundational figure of modern physics.
 45 | F1=0.0000 | David Beckham
 49 | F1=0.0000 | The article mentions David Beckham, an Italian astronomer, physicist, mathematician,
 53 | F1=0.0000 | John Beverley
 57 | F1=0.0000 | The name of the scientist widely acclaimed as the foundational figure of modern physics is John
 61 | F1=0.0000 | The scientist widely acclaimed as the foundational figure of